<img src="https://raw.githubusercontent.com/senanurcetin/enveda-casmi26/main/site/assets/nexus-midnight.svg" alt="NEXUS official identity" width="360">

# CASMI26: Exact Public-Example Overlap Audit

**NEXUS / MOLECULAR IDENTIFICATION RESEARCH** · Senanur Cetin · Noncommercial research

The historical profile found coarse-key matches between public examples and training spectra. A rounded precursor, peak count and first peak cannot establish exact overlap. This notebook reruns that question on the attached public files, comparing **every stored fragment m/z and normalized intensity, in order, plus the unrounded precursor value**.

It separately checks adduct and ionization-mode agreement. Equality of stored numeric content does not independently authenticate acquisition identity. The mounted public example file is distinct from the competition's hidden evaluation set.

Only aggregate counts, input file SHA-256 hashes, source-code hashes and software versions are exported. Raw molecule/spectrum identifiers, SMILES, peak arrays, models and predictions are not published. No model training, performance scoring or competition submission occurs. CPU only; internet disabled.


In [ ]:
# NEXUS chart defaults; no remote assets or paid services needed for computation.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    'figure.facecolor': '#0B0F19', 'axes.facecolor': '#061524',
    'savefig.facecolor': '#0B0F19', 'text.color': '#EAF4FF',
    'axes.labelcolor': '#B9C5CE', 'axes.edgecolor': '#52667D',
    'xtick.color': '#B9C5CE', 'ytick.color': '#B9C5CE',
    'grid.color': '#29384D', 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.prop_cycle': cycler(color=['#00E0FF', '#8F9BFF', '#4EA8FF', '#DDB26F']),
    'figure.dpi': 120, 'font.size': 10, 'axes.spines.top': False,
    'axes.spines.right': False,
})
print('NEXUS research edition. Printed historical tables are not new leaderboard results.')


## 1. Load the versioned audit code

In [ ]:
# Source files are bundled from the tracked repository; no internet/install step.
import json, sys
from pathlib import Path
bundle = Path('/kaggle/working/casmi_overlap_source')
sources = json.loads('{"src/casmi/validate/spectrum_overlap.py": "\\"\\"\\"Exact-peak overlap helpers for CASMI validation leakage checks.\\n\\nA precursor mass, number of peaks and first m/z are a *bucket*, not evidence\\nof a duplicate spectrum. Only full arrays, including intensities, can confirm\\na same-spectrum match. This module deliberately has no numerical dependencies.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom collections.abc import Sequence\\n\\n\\ndef coarse_spectrum_bucket(precursor_mz: float, ms2_mzs: Sequence) -> tuple | None:\\n    \\"\\"\\"Cheap candidate retrieval; NEVER use the bucket as a match on its own.\\"\\"\\"\\n    if ms2_mzs is None or len(ms2_mzs) == 0:\\n        return None\\n    try:\\n        precursor, first = float(precursor_mz), float(ms2_mzs[0])\\n    except (TypeError, ValueError):\\n        return None\\n    if not math.isfinite(precursor) or not math.isfinite(first):\\n        return None\\n    return round(precursor, 4), len(ms2_mzs), round(first, 4)\\n\\n\\ndef full_spectrum_match(\\n    precursor_a, mz_a: Sequence, intensities_a: Sequence,\\n    precursor_b, mz_b: Sequence, intensities_b: Sequence,\\n) -> bool:\\n    \\"\\"\\"Require full precursor, fragment m/z and normalized intensity equality.\\n\\n    Converts array scalar dtypes to Python float without rounding. An equal\\n    coarse bucket with shifted *any* peak or intensity is not a verified match.\\n    \\"\\"\\"\\n    if mz_a is None or mz_b is None or intensities_a is None or intensities_b is None:\\n        return False\\n    if len(mz_a) == 0 or len(mz_b) == 0:\\n        return False\\n    if len(mz_a) != len(mz_b) or len(intensities_a) != len(intensities_b):\\n        return False\\n    if len(mz_a) != len(intensities_a):\\n        return False\\n    try:\\n        values_a = [float(precursor_a)] + [float(x) for x in mz_a] + [float(x) for x in intensities_a]\\n        values_b = [float(precursor_b)] + [float(x) for x in mz_b] + [float(x) for x in intensities_b]\\n    except (TypeError, ValueError):\\n        return False\\n    if not all(math.isfinite(x) for x in values_a + values_b):\\n        return False\\n    return values_a == values_b\\n", "src/casmi/validate/overlap_audit.py": "\\"\\"\\"Aggregate-only, exact-content public-example overlap audit.\\n\\nRaw identifiers and peak arrays stay in memory and are never exported. Equality\\nof stored numeric arrays is distinct from proof of acquisition provenance.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections import defaultdict\\n\\nfrom .spectrum_overlap import coarse_spectrum_bucket, full_spectrum_match\\n\\n\\ndef _spectrum(row):\\n    return (row.get(\\"precursor_mz\\"), row.get(\\"ms2_mzs\\"),\\n            row.get(\\"ms2_normalized_intensities\\"))\\n\\n\\ndef _identifier(value):\\n    if value is None or not str(value).strip():\\n        raise ValueError(\\"Public example identifiers must be present\\")\\n    return str(value)\\n\\n\\nclass PublicOverlapAudit:\\n    def __init__(self, public_rows):\\n        self.targets = list(public_rows)\\n        if not self.targets:\\n            raise ValueError(\\"Public example input must not be empty\\")\\n        self.buckets = defaultdict(list)\\n        self.molecules = defaultdict(set)\\n        seen = set()\\n        for index, row in enumerate(self.targets):\\n            sid = _identifier(row.get(\\"spectrum_id\\"))\\n            molecule = _identifier(row.get(\\"molecule_id\\"))\\n            if sid in seen:\\n                raise ValueError(\\"Duplicate public spectrum identifiers\\")\\n            seen.add(sid)\\n            spectrum = _spectrum(row)\\n            if not full_spectrum_match(*spectrum, *spectrum):\\n                raise ValueError(\\"Public spectrum must have finite aligned nonempty arrays\\")\\n            self.buckets[coarse_spectrum_bucket(spectrum[0], spectrum[1])].append(index)\\n            self.molecules[molecule].add(index)\\n        self.precursors = {float(row[\\"precursor_mz\\"]) for row in self.targets}\\n        self.scanned = 0\\n        self.candidate_rows = 0\\n        self.coarse_pairs = 0\\n        self.exact_pairs = 0\\n        self.false_pairs = 0\\n        self.matched = set()\\n        self.metadata_matched = set()\\n        self.library_matches = defaultdict(set)\\n        self.metadata_disagreement_pairs = 0\\n\\n    def scan_batch(self, candidate_rows, *, scanned_count=None):\\n        \\"\\"\\"Count all scanned rows, considering only exact-precursor candidates.\\n\\n        A caller can filter Arrow batches before converting peak lists to Python.\\n        `scanned_count` must include filtered-out rows; finish checks full coverage.\\n        \\"\\"\\"\\n        rows = list(candidate_rows)\\n        count = len(rows) if scanned_count is None else scanned_count\\n        if isinstance(count, bool) or not isinstance(count, int) or count < len(rows):\\n            raise ValueError(\\"Scanned row count must cover every considered row\\")\\n        self.scanned += count\\n        for row in rows:\\n            try:\\n                precursor = float(row.get(\\"precursor_mz\\"))\\n            except (TypeError, ValueError):\\n                continue\\n            if precursor not in self.precursors:\\n                continue\\n            self.candidate_rows += 1\\n            spectrum = _spectrum(row)\\n            bucket = coarse_spectrum_bucket(spectrum[0], spectrum[1])\\n            for index in self.buckets.get(bucket, ()):\\n                self.coarse_pairs += 1\\n                target = self.targets[index]\\n                if not full_spectrum_match(*spectrum, *_spectrum(target)):\\n                    self.false_pairs += 1\\n                    continue\\n                library = row.get(\\"ingest_lib\\")\\n                if not isinstance(library, str) or not library.strip():\\n                    raise ValueError(\\"Matching training spectra need a source-library label\\")\\n                self.exact_pairs += 1\\n                self.matched.add(index)\\n                self.library_matches[library].add(index)\\n                aligned = all(target.get(key) is not None and row.get(key) == target[key]\\n                              for key in (\\"adduct\\", \\"ionization_mode\\"))\\n                if aligned:\\n                    self.metadata_matched.add(index)\\n                else:\\n                    self.metadata_disagreement_pairs += 1\\n\\n    def finish(self, *, expected_train_rows):\\n        if (isinstance(expected_train_rows, bool) or not isinstance(expected_train_rows, int)\\n                or expected_train_rows <= 0 or self.scanned != expected_train_rows):\\n            raise ValueError(\\"A complete scan of the declared training row count is required\\")\\n        return {\\n            \\"schema_version\\": 1,\\n            \\"status\\": \\"COMPLETE_CONTENT_AUDIT\\",\\n            \\"matching_method\\": \\"exact stored precursor and full ordered m/z/intensity numeric arrays; no rounding or tolerance\\",\\n            \\"metadata_fields\\": [\\"adduct\\", \\"ionization_mode\\"],\\n            \\"train_rows_scanned\\": self.scanned,\\n            \\"public_spectra\\": len(self.targets),\\n            \\"public_molecules\\": len(self.molecules),\\n            \\"exact_precursor_candidate_rows\\": self.candidate_rows,\\n            \\"coarse_candidate_pairs_after_precursor_filter\\": self.coarse_pairs,\\n            \\"coarse_false_match_pairs\\": self.false_pairs,\\n            \\"exact_matching_pairs\\": self.exact_pairs,\\n            \\"content_matched_spectra\\": len(self.matched),\\n            \\"metadata_consistent_matched_spectra\\": len(self.metadata_matched),\\n            \\"metadata_disagreement_pairs\\": self.metadata_disagreement_pairs,\\n            \\"fully_content_matched_molecules\\": sum(indices <= self.matched for indices in self.molecules.values()),\\n            \\"fully_metadata_consistent_molecules\\": sum(indices <= self.metadata_matched for indices in self.molecules.values()),\\n            \\"content_matches_by_library\\": {key: len(value) for key, value in sorted(self.library_matches.items())},\\n            \\"limitations\\": [\\n                \\"Source-library match counts overlap and must not be summed as unique public spectra.\\",\\n                \\"Stored numerical equality does not independently prove common acquisition identity.\\",\\n                \\"This audit concerns mounted public examples, not the hidden competition evaluation set.\\",\\n                \\"No model evaluation, training or competition submission was performed.\\",\\n            ],\\n        }\\n", "scripts/audit_public_overlap.py": "\\"\\"\\"Read-only, bounded-memory exact public/train overlap audit on authorized Parquet files.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport platform\\nimport sys\\nfrom datetime import UTC, datetime\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parents[1]\\nsys.path.insert(0, str(ROOT / \\"src\\"))\\nfrom casmi.validate.overlap_audit import PublicOverlapAudit  # noqa: E402\\n\\nPEAKS = [\\"precursor_mz\\", \\"ms2_mzs\\", \\"ms2_normalized_intensities\\", \\"adduct\\", \\"ionization_mode\\"]\\n\\n\\ndef file_record(path):\\n    digest = hashlib.sha256()\\n    with path.open(\\"rb\\") as source:\\n        for block in iter(lambda: source.read(8 * 1024 * 1024), b\\"\\"):\\n            digest.update(block)\\n    return {\\"bytes\\": path.stat().st_size, \\"sha256\\": digest.hexdigest()}\\n\\n\\ndef run_audit(train_path, public_path, *, batch_size=8192, progress=None):\\n    import pyarrow as pa\\n    import pyarrow.compute as pc\\n    import pyarrow.parquet as pq\\n\\n    if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size <= 0:\\n        raise ValueError(\\"Batch size must be a positive integer\\")\\n    train_path, public_path = Path(train_path), Path(public_path)\\n    if train_path.resolve() == public_path.resolve():\\n        raise ValueError(\\"Training and public-example paths must be different\\")\\n    train = pq.ParquetFile(train_path)\\n    public = pq.ParquetFile(public_path)\\n    for parquet, required in ((train, PEAKS + [\\"ingest_lib\\"]),\\n                              (public, PEAKS + [\\"spectrum_id\\", \\"molecule_id\\"])):\\n        if not set(required) <= set(parquet.schema_arrow.names):\\n            raise ValueError(\\"Missing required audit columns\\")\\n    # Public inputs should be small. Refuse accidental full-train target materialization.\\n    if public.metadata.num_rows > 100_000:\\n        raise ValueError(\\"Public example input exceeds the 100,000-row safety ceiling\\")\\n    targets = public.read(columns=PEAKS + [\\"spectrum_id\\", \\"molecule_id\\"]).to_pylist()\\n    audit = PublicOverlapAudit(targets)\\n    precursors = pa.array(sorted(audit.precursors), type=pa.float64())\\n    for index, batch in enumerate(train.iter_batches(batch_size=batch_size, columns=PEAKS + [\\"ingest_lib\\"])):\\n        mask = pc.is_in(pc.cast(batch.column(\\"precursor_mz\\"), pa.float64()), value_set=precursors)\\n        # Exact equality of precursors is necessary for any full-content match.\\n        # Arrow filters before list materialization; peak arrays of decoys stay in Arrow.\\n        selected = batch.filter(mask).to_pylist()\\n        audit.scan_batch(selected, scanned_count=batch.num_rows)\\n        if progress is not None and index % 25 == 0:\\n            progress(audit.scanned, train.metadata.num_rows)\\n    result = audit.finish(expected_train_rows=train.metadata.num_rows)\\n    result[\\"checked_at\\"] = datetime.now(UTC).isoformat()\\n    result[\\"inputs\\"] = {\\"train.parquet\\": file_record(train_path), \\"test.parquet\\": file_record(public_path)}\\n    result[\\"audit_sources\\"] = {\\n        name: hashlib.sha256((ROOT / name).read_text(encoding=\\"utf-8\\").encode()).hexdigest()\\n        for name in (\\"src/casmi/validate/spectrum_overlap.py\\", \\"src/casmi/validate/overlap_audit.py\\", \\"scripts/audit_public_overlap.py\\")\\n    }\\n    result[\\"environment\\"] = {\\"python\\": platform.python_version(), \\"pyarrow\\": importlib.metadata.version(\\"pyarrow\\")}\\n    result[\\"batch_size\\"] = batch_size\\n    return result\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--public\\", type=Path, required=True)\\n    parser.add_argument(\\"--output\\", type=Path, required=True)\\n    parser.add_argument(\\"--batch-size\\", type=int, default=8192)\\n    args = parser.parse_args()\\n    if args.output.exists():\\n        raise FileExistsError(\\"Refusing to overwrite an existing audit report\\")\\n    result = run_audit(args.train, args.public, batch_size=args.batch_size,\\n                       progress=lambda count, total: print(f\\"Scanned {count:,}/{total:,} training rows\\", flush=True))\\n    args.output.parent.mkdir(parents=True, exist_ok=True)\\n    with args.output.open(\\"x\\", encoding=\\"utf-8\\") as destination:\\n        json.dump(result, destination, indent=2, allow_nan=False)\\n        destination.write(\\"\\\\n\\")\\n    print(json.dumps(result, indent=2, allow_nan=False))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for name, text in sources.items():
    path = bundle / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
sys.path.insert(0, str(bundle / 'src'))
sys.path.insert(0, str(bundle / 'scripts'))
from audit_public_overlap import run_audit
print('Loaded content-audit source; no model or prediction code is invoked.')


## 2. Scan all training rows and verify full content

Arrow batches limit memory. Exact precursor equality is a necessary prefilter, followed by full-array comparison. The scanner fails if the visited row count does not match the Parquet footer. Input hashes describe the exact mounted files used by this run.

In [ ]:
# There must be exactly one attached competition train/test pair.
pairs = [path for path in Path('/kaggle/input').rglob('train.parquet')
         if path.with_name('test.parquet').is_file()]
if len(pairs) != 1:
    raise RuntimeError('Expected exactly one attached competition train/test pair')
result = run_audit(pairs[0], pairs[0].with_name('test.parquet'), batch_size=8192,
                   progress=lambda count, total: print(f'Scanned {count:,}/{total:,} train rows', flush=True))
result['execution'] = {
    'platform': 'Kaggle',
    'competition_source': 'enveda-CASMI26-molecule-id-mass-spectra',
    'input_scope': 'mounted public train.parquet and test.parquet; not hidden evaluation',
}
with Path('/kaggle/working/public_overlap_audit.json').open('x', encoding='utf-8') as output:
    json.dump(result, output, indent=2, allow_nan=False)
    output.write('\n')
print(json.dumps(result, indent=2, allow_nan=False))


## 3. Read the result without overstating it

In [ ]:
labels = ['Public examples', 'Exact numeric content', '+ Adduct/mode agreement']
counts = [result['public_spectra'], result['content_matched_spectra'],
          result['metadata_consistent_matched_spectra']]
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(labels[::-1], counts[::-1], color=['#8F9BFF', '#00E0FF', '#4EA8FF'])
ax.bar_label(bars, padding=5)
ax.set_xlim(0, max(counts) * 1.15)
ax.set_xlabel('Public-example spectra (unique target rows)')
ax.set_title('Executed exact-content overlap audit — not a model evaluation')
plt.tight_layout()
plt.show()


A matched public example is unsuitable as an independent unseen-spectrum holdout. This finding says nothing by itself about hidden-test contamination, model ranking quality or prospective chemical identification.

Source-library match counts are **not additive**: one public spectrum can match multiple training records/libraries. A molecule is fully matched only if all its public spectra have matches. A matching content pair with conflicting acquisition metadata is counted separately.

An exact comparison is deliberately stricter than tolerance-based spectral similarity. A stored floating-point difference or a changed peak order produces a non-match even if the underlying acquisition was related. Inspect the JSON for exact/metadata distinctions and denominator coverage; do not substitute rounded-key matches for exact matches.

[Repository methodology and model card](https://github.com/senanurcetin/enveda-casmi26/blob/main/MODEL_CARD.md). Historical model scores remain unchanged by this data audit.
